# 22.5 — Beam search

Beam search keeps the best $k$ partial candidates and deliberately forgets the rest.

A* ranks a frontier; beam search adds a hard memory budget. It builds sequences layer by layer and keeps only the top candidates, so it is useful but not an optimality proof.

Save a copy to Drive to edit.

In [ ]:
import math
import random
import matplotlib.pyplot as plt

SEED = 225
random.seed(SEED)


def generate_sequence_tree(depth, branching, delayed_branch=1, delayed_bonus=8, variable_lengths=False):
    alphabet = [chr(ord("A") + i) for i in range(branching)]
    base = {letter: branching - i for i, letter in enumerate(alphabet)}

    def successors(prefix):
        if len(prefix) >= depth:
            return []
        out = []
        for i, letter in enumerate(alphabet):
            child = prefix + letter
            step_score = base[letter]
            if len(prefix) == depth - 1 and prefix.startswith(alphabet[delayed_branch]):
                step_score = step_score + delayed_bonus
            if variable_lengths and letter == alphabet[-1] and len(prefix) >= 1:
                child = child + "!"
            out.append((child, step_score))
        return out

    return successors


def make_beam_instances():
    return [
        {"name": "D1 tiny two-layer", "depth": 2, "branching": 2, "delayed_branch": 1, "bonus": 6, "widths": [1, 2], "variable": False},
        {"name": "D2 wider branching", "depth": 3, "branching": 4, "delayed_branch": 2, "bonus": 8, "widths": [1, 2, 4], "variable": False},
        {"name": "D3 deeper delayed reward", "depth": 5, "branching": 4, "delayed_branch": 3, "bonus": 14, "widths": [1, 2, 4], "variable": False},
        {"name": "D4 variable length scoring", "depth": 6, "branching": 5, "delayed_branch": 3, "bonus": 16, "widths": [1, 3, 5], "variable": True},
        {"name": "D5 largest deceptive delayed payoff", "depth": 8, "branching": 6, "delayed_branch": 5, "bonus": 28, "widths": [1, 2, 4, 8], "variable": True},
    ]


## The concept, built once (D1)

At depth $t$, beam search expands every candidate in $B_t$, sorts by score $S(x)$, and keeps the top $k$. Width $1$ is greedy; a wider beam can preserve a delayed payoff branch.

In [ ]:
def beam_search(successors, score, width, max_depth, normalize=False):
    beam = [""]
    generated = 0
    history = []

    for depth in range(max_depth):
        candidates = []
        for prefix in beam:
            for child, step_score in successors(prefix):
                generated = generated + 1
                candidates.append(child)
        if not candidates:
            break
        ranked = sorted(candidates, key=lambda x: score(x, normalize=normalize), reverse=True)
        kept = ranked[:width]
        pruned = ranked[width:]
        history.append({
            "depth": depth + 1,
            "expanded": list(beam),
            "kept": kept,
            "pruned": pruned,
        })
        beam = kept

    best = max(beam, key=lambda x: score(x, normalize=normalize))
    return {
        "best": best,
        "quality": score(best, normalize=normalize),
        "generated": generated,
        "history": history,
    }


def lesson_score(seq, normalize=False):
    raw_scores = {
        "A": 5,
        "B": 4,
        "AA": 5,
        "AB": 6,
        "BA": 10,
        "BB": 3,
        "SHORT": 6,
        "LONG": 9,
    }
    raw = raw_scores.get(seq, 0)
    if normalize:
        denom = 2 if seq == "SHORT" else 5 if seq == "LONG" else max(1, len(seq))
        return raw / denom
    return raw


def lesson_successors(prefix):
    graph = {
        "": [("A", 5), ("B", 4)],
        "A": [("AA", 0), ("AB", 1)],
        "B": [("BA", 6), ("BB", -1)],
    }
    return graph.get(prefix, [])


The exact lesson checks are width-$1$ final $AB=6$, width-$2$ final $BA=10$, first layer $A=5$ over $B=4$, successor burst $k\cdot b=5\cdot20=100$, and normalization $6/2=3.0$ versus $9/5=1.8$.

In [ ]:
width1 = beam_search(lesson_successors, lesson_score, width=1, max_depth=2)
width2 = beam_search(lesson_successors, lesson_score, width=2, max_depth=2)
successor_burst = 5 * 20
short_norm = lesson_score("SHORT", normalize=True)
long_norm = lesson_score("LONG", normalize=True)

assert lesson_score("A") == 5
assert lesson_score("B") == 4
assert width1["best"] == "AB"
assert width1["quality"] == 6
assert width2["best"] == "BA"
assert width2["quality"] == 10
assert successor_burst == 100
assert short_norm == 3.0
assert long_norm == 1.8


## The dataset ladder

D1–D5 are generated sequence trees, not a hardcoded lattice. Size, branching, delayed payoff, and variable-length scoring all grow so D5 can prune the true winner under narrow beams.

In [ ]:
instances = make_beam_instances()
for item in instances:
    total_candidates = sum(item["branching"] ** depth for depth in range(1, item["depth"] + 1))
    print(item["name"], "depth", item["depth"], "branching", item["branching"], "candidates", total_candidates)


In [ ]:
def make_score(successors):
    cache = {"": 0}

    def raw_score(seq, normalize=False):
        if seq in cache:
            raw = cache[seq]
        else:
            parent = seq[:-1]
            if seq.endswith("!"):
                parent = seq[:-2]
            raw = -math.inf
            for child, step in successors(parent):
                if child == seq:
                    raw = raw_score(parent) + step
            cache[seq] = raw
        if normalize:
            length = max(1, len(seq.replace("!", "")))
            return raw / length
        return raw

    return raw_score


def exhaustive_best(successors, score, max_depth, normalize=False):
    layer = [""]
    all_nodes = []
    for depth in range(max_depth):
        nxt_layer = []
        for prefix in layer:
            for child, _ in successors(prefix):
                nxt_layer.append(child)
                all_nodes.append(child)
        layer = nxt_layer
    best = max(all_nodes, key=lambda x: score(x, normalize=normalize))
    return best, score(best, normalize=normalize), len(all_nodes)

rows = []
for item in instances:
    successors = generate_sequence_tree(item["depth"], item["branching"], item["delayed_branch"], item["bonus"], item["variable"])
    score = make_score(successors)
    optimal = exhaustive_best(successors, score, item["depth"], normalize=item["variable"])
    width = item["widths"][-2]
    result = beam_search(successors, score, width=width, max_depth=item["depth"], normalize=item["variable"])
    rows.append({
        "rung": item["name"].split()[0],
        "width": width,
        "beam_quality": result["quality"],
        "optimal_quality": optimal[1],
        "quality_gap": optimal[1] - result["quality"],
        "generated": result["generated"],
        "exhaustive_nodes": optimal[2],
        "history": result["history"],
    })

for row in rows:
    print(row)


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, row in zip(axes[0], rows):
    ax.set_title(row["rung"])
    kept_counts = [len(step["kept"]) for step in row["history"]]
    pruned_counts = [len(step["pruned"]) for step in row["history"]]
    xs = list(range(1, len(kept_counts) + 1))
    ax.bar(xs, kept_counts, color="lime", label="kept")
    ax.bar(xs, pruned_counts, bottom=kept_counts, color="lightgray", label="pruned")
    ax.set_xlabel("depth")

labels = [row["rung"] for row in rows]
axes[1, 0].plot(labels, [row["quality_gap"] for row in rows], marker="o", color="red")
axes[1, 0].set_title("quality gap vs size")
axes[1, 1].plot(labels, [row["generated"] for row in rows], marker="o", label="beam generated")
axes[1, 1].plot(labels, [row["exhaustive_nodes"] for row in rows], marker="o", label="exhaustive nodes")
axes[1, 1].set_title("generated nodes vs size")
axes[1, 1].legend()
for ax in axes[1, 2:]:
    ax.axis("off")
plt.tight_layout()


## Pitfall on D5: beam search is not optimal

A narrow beam can prune the delayed winner before its payoff appears. The fix is to widen the beam, and when candidate lengths vary, compare normalized scores rather than raw totals.

In [ ]:
d5 = instances[-1]
successors = generate_sequence_tree(d5["depth"], d5["branching"], d5["delayed_branch"], d5["bonus"], d5["variable"])
score = make_score(successors)
optimal = exhaustive_best(successors, score, d5["depth"], normalize=True)
narrow = beam_search(successors, score, width=1, max_depth=d5["depth"], normalize=True)
wide = beam_search(successors, score, width=8, max_depth=d5["depth"], normalize=True)
print("optimal", optimal[0], optimal[1])
print("narrow", narrow["best"], narrow["quality"], "gap", optimal[1] - narrow["quality"])
print("wide", wide["best"], wide["quality"], "gap", optimal[1] - wide["quality"])


## Evaluate it + Practice

- Main metric: solution quality gap and candidates generated.
- No-skill baseline: greedy width 1.
- Cheap sanity check: rerun D1 and verify the hand-trace number from the lesson exactly.
- Ablation: reduce width from 8 to 1 or turn off length normalization on D4/D5.
- Failure signals: large successor bursts hidden by a small retained beam or a positive gap reported as optimal.

Practice prompts:
1. Change one D3 obstacle, edge, or score parameter and predict which nodes change before running.
2. Add one more D5 deceptive branch and compare the metric table.
3. Write a one-sentence rule for when the pitfall would be dangerous in production.